# Modèle avancé avec Tensorflow

# Entraînement du modèle

In [4]:
# Example code to display the version of all imports
import pkg_resources

# List of package names
packages = ['pandas',
           'nltk',
           'scikit-learn',
           'tensorflow']

for package in packages:
    version = pkg_resources.get_distribution(package).version
    print(f"{package} == {version}")
    print()

pandas == 1.5.3

nltk == 3.8.1

scikit-learn == 1.2.2

tensorflow == 2.15.0



In [1]:
'''
This script uses the following python and packages versions :

python == 3.10.9 | packaged by Anaconda.

pandas == 1.5.3

nltk == 3.8.1

scikit-learn == 1.2.2

tensorflow == 2.15.0

'''

import re
import string
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Activation, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.metrics import AUC
import pickle

# Removal of unnecessary imports and warnings configuration
import warnings
warnings.filterwarnings("ignore")


def cleaning_stopwords(text):
    """
    Removes stopwords from the text.
    """
    return " ".join([word for word in str(text).split() if word not in STOPWORDS])

def cleaning_punctuations(text):
    """
    Removes punctuation from the text.
    """
    translator = str.maketrans('', '', punctuations_list)
    return text.translate(translator)

def cleaning_repeating_char(text):
    """
    Replaces repeating characters in the text.
    """
    return re.sub(r'(.)\1+', r'\1', text)

def cleaning_email(data):
    """
    Removes email addresses from the text.
    """
    return re.sub('@[^\s]+', ' ', data)

def cleaning_URLs(data):
    """
    Removes URLs from the text.
    """
    return re.sub('((www\.[^\s]+)|(https?://[^\s]+))',' ',data)

def cleaning_numbers(data):
    """
    Removes numbers from the text.
    """
    return re.sub('[0-9]+', '', data)

def stemming_on_text(data):
    """
    Applies stemming to the text.
    """
    text = [st.stem(word) for word in data]
    return data

def lemmatizer_on_text(data):
    """
    Applies lemmatization to the text.
    """
    text = [lm.lemmatize(word) for word in data]
    return data

# Data reading
data = pd.read_csv("data_raw/eda-twitter-sentiment-analysis-using-nn.csv", encoding = "ISO-8859-1", engine="python")
data.columns = ["label", "time", "date", "query", "username", "text"]

# Selecting text and label columns
data=data[['text','label']]

# Replacing label '4' with '1'
data['label'][data['label']==4]=1

# Separating positive and negative tweets
data_pos = data[data['label'] == 1]
data_neg = data[data['label'] == 0]

# Keeping only 25% of the data for quick run
data_pos = data_pos.iloc[:int(20000)]
data_neg = data_neg.iloc[:int(20000)]

# Merging positive and negative tweets
data = pd.concat([data_pos, data_neg])

# Converting uppercase to lowercase
data['text']=data['text'].str.lower()

# Removing English stopwords
STOPWORDS = set(stopwords.words('english'))

# Removing stop words
data['text'] = data['text'].apply(lambda text: cleaning_stopwords(text))

# Removing punctuation
punctuations_list = string.punctuation
data['text']= data['text'].apply(lambda x: cleaning_punctuations(x))

# Removing repeated characters
data['text'] = data['text'].apply(lambda x: cleaning_repeating_char(x))

# Removing emails
data['text']= data['text'].apply(lambda x: cleaning_email(x))

# Removing URLs
data['text'] = data['text'].apply(lambda x: cleaning_URLs(x))

# Removing numbers
data['text'] = data['text'].apply(lambda x: cleaning_numbers(x))

# Tokenization of tweets
tokenizer = nltk.tokenize.RegexpTokenizer(r'\w+')
data['text'] = data['text'].apply(tokenizer.tokenize)

# Stemming
st = nltk.PorterStemmer()
data['text']= data['text'].apply(lambda x: stemming_on_text(x))

# Lemmatization
lm = nltk.WordNetLemmatizer()
data['text'] = data['text'].apply(lambda x: lemmatizer_on_text(x))

# Sequential encoding of tokens
max_len = 500
tok = Tokenizer(num_words=2000)
tok.fit_on_texts(data.text)
sequences = tok.texts_to_sequences(data.text)
sequences_matrix = sequence.pad_sequences(sequences,maxlen=max_len)

# Splitting data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(sequences_matrix, data.label, test_size=0.3, random_state=2)

# Model compilation
def tensorflow_based_baseline_model():
    inputs = Input(name='inputs', shape=[max_len])
    layer = Embedding(2000, 50, input_length=max_len)(inputs)
    layer = Flatten()(layer)
    layer = Dense(256, name='FC1')(layer)
    layer = Activation('relu')(layer)
    layer = Dropout(0.5)(layer)
    layer = Dense(1, name='out_layer')(layer)
    layer = Activation('sigmoid')(layer)
    model = Model(inputs=inputs, outputs=layer)
    return model

# Model instantiation and configuration
auc_roc = AUC(curve='ROC')
model = tensorflow_based_baseline_model()
model.compile(loss='binary_crossentropy', optimizer=RMSprop(), metrics=[auc_roc])

# Model training
history = model.fit(X_train, Y_train, batch_size=80, epochs=6, validation_split=0.1)
print('Training finished !!')

# Model testing
accr1 = model.evaluate(X_test, Y_test)
print('Test set\n  Area under the ROC Curve: {:0.2f}'.format(accr1[1]))

# Saves the architecture, weights, and training config of the model
model.save('model_full.h5')  
print("Model saved to 'model_full.h5'")

# Save tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tok, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Tokenizer saved to 'tokenizer.pickle'")


NameError: name 'pd' is not defined